# Анализ результатов A/B-теста онлайн-кинотеатра

## Цель проекта
Подготовить данные к анализу результатов A/B-теста и оценить влияние новой функциональности на поведение пользователей.

## Стек
- Python
- Pandas
- NumPy
- SciPy
- Matplotlib

## Задачи
1. Проверка качества данных.
2. Поиск пользователей, попавших в несколько групп.
3. Очистка выборки.
4. Расчёт метрик.
5. Проверка статистических гипотез.
6. Формирование рекомендаций для бизнеса.

### Задача 1

Импортируйте файл Event_click.xlsx.\
В нем содержатся результаты A/B теста, который проводился в онлайн-кинотеатре. В рамках эксперимента тестировалась новая рекомендательная система, цель которой - убедить пользователя просмотреть тот или иной тайтл.\

Описание данных:
- id_client - уникальный идентификатор клиента
- dtime_event - дата совершения события (временная часть опущена для облегчения данных)
- flag_group - флаг принадлежности к тестовой группе (1 - тест, 0 -контроль).
- event - залогированное клиентское событие (play - пригрывание плеера, click - клик мышкой, scroll - прокручивание страницы)

Необходимо подготовить данные к анализу результатов A/B теста.\
Для этого:
1. Определите количество нуллов в рамках каждого столбца (и их пересечений) и исключите все записи, в которых отсутствует информация хотя бы в одном поле.
2. Выявите клиентов, которые в течение эксперимента попадали в разные группы (*Подсказка: рассчитайте количество уникальных значений поля flag_group на каждого клиента*)
3. Исключите этих клиентов из выборки.
4. Оцените общее количество исключений (относительно всей выборки)

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats 
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
df_bad = pd.read_excel("data/Event_click.xlsx")
df_bad

In [ ]:
#Определите количество нуллов в рамках каждого столбца (и их пересечений) и исключите все записи,
#в которых отсутствует информация хотя бы в одном поле

df = df_bad.dropna()
print(len(df['id_client']))
print(df)

In [ ]:
#Выявите клиентов, которые в течение эксперимента попадали в разные группы 
#(Подсказка: рассчитайте количество уникальных значений поля flag_group на каждого клиента)

df_gr_cl = df.groupby('id_client')['flag_group'].nunique().reset_index()
print(df_gr_cl)


In [ ]:
#Исключите этих клиентов из выборки.
df_gr_good = df_gr_cl[df_gr_cl['flag_group']==1]
print(df_gr_good)


In [ ]:
df_good = df[df['id_client'].isin(df_gr_good['id_client'])]
df_good

In [ ]:
#Оцените общее количество исключений (относительно всей выборки)
len(df_bad['id_client']) - len(df_good['id_client'])

## Задача 2

Задача новой рекомендательной системы заключается в том, чтобы повысить процент людей, которые при заходе на нашу страницу (то есть при совершении любого действия) перейдут в плеер (то есть совершат действие play).

Рассчитайте отдельно для *контрольной* и *тестовой* группы процент пользователей, которые совершили хотя бы одно действие play. Можете ли вы сделать вывод о значимом улучшении конверсии в просмотр?

Обратите внимание, что в Задании 2 мы работаем с очищенными данными, полученными в результате выполнения Задания 1.

Алгоритм решения:

+ создайте в данных новую колонку flag_play, которая принимает значение 1, если совершаемое действие - ‘play’ и 0 для любого другого действия


+ сгруппируйте данные по двум столбцам одновременно - пользователь и его группа - и посчитайте максимальное значение по колонке flag_play (если пользователь хотя бы раз играл, то максимальное значение будет 1, если ни разу, то окажется 0), назовем его max_flag


+ возьмите среднее значение max_flag  для тестовой и контрольной групп - отличаются ли значения?


+ проведите t-тест по колонке max_flag для тестовой и контрольной групп, чтобы оценить стат. значимость разницы в средних значениях между ними. Воспользуйтесь функцией ttest_ind.

In [ ]:
df_good.head()

In [ ]:
df_good = df_good.copy()
df_good

In [ ]:
#создайте в данных новую колонку flag_play, которая принимает значение 1, если совершаемое действие - ‘play’ 
#и 0 для любого другого действия

df_good['flag_play'] = np.where(df_good['event'] == 'play', 1, 0)
df_good

In [ ]:
#сгруппируйте данные по двум столбцам одновременно - пользователь и его группа - и посчитайте максимальное значение 
#по колонке flag_play (если пользователь хотя бы раз играл, то максимальное значение будет 1, если ни разу, то окажется 0), 
#назовем его max_flag


df_good_gr = df_good.groupby(['id_client', 'flag_group']).agg(max_flag = ('flag_play', 'max')).reset_index()
print(df_good_gr)
df_good_gr.dtypes

## Формирование контрольной и тестовой групп

После подготовки данных разделим пользователей на контрольную и тестовую группы для дальнейшего сравнения ключевых метрик.а.

In [ ]:
test_group = df_good_gr[df_good_gr['flag_group'] == 1]
control_group = df_good_gr[df_good_gr['flag_group'] == 0]

print(test_group)
print(control_group)

In [ ]:
#возьмите среднее значение max_flag для тестовой и контрольной групп - отличаются ли значения?

avg_test_group = df_good_gr[df_good_gr['flag_group'] == 1]['max_flag'].mean()
avg_control_group = df_good_gr[df_good_gr['flag_group'] == 0]['max_flag'].mean()

print(avg_test_group)
print(avg_control_group)


In [ ]:
from scipy.stats import ttest_ind

In [ ]:
#проведите t-тест по колонке max_flag для тестовой и контрольной групп, чтобы оценить стат. значимость разницы в средних 
#значениях между ними. Воспользуйтесь функцией ttest_ind.

s, p = stats.ttest_ind(test_group['max_flag'], control_group['max_flag'])
print(s)
print(p)

## Задача 3

Рассчитайте DAU и WAU по имеющимся данным.\
Допустим, активностью пользователя является любая активность (совершение любого действия, хотя бы одного).\

1. Рассчитайте и визуализируйте DAU (по оси Х - дни)
2. Рассчитайте и визуализируйте WAU (по оси Х - недели) (*исключите неполные недели*)
3. Соедините DAU и WAU и визуализируйте их на одном графике (по оси Х - дни)

Какие выводы можно сделать на основании имеющихся данных?

*Подсказка: чтобы определить номер недели в году используйте метод isocalendar()*\
*Ознакомиться с документацией можно по ссылке: https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.isocalendar.html*

In [ ]:
print(df_good)
df_good.dtypes

In [ ]:
iso_calendar = pd.to_datetime(df_good['dtime_event']).dt.isocalendar()

df_good = pd.concat([df_good, iso_calendar], axis=1)
df_good

In [ ]:
#Рассчитайте и визуализируйте DAU (по оси Х - дни)

df_dau = df_good.groupby(['day', 'dtime_event']).agg(DAU = ('id_client', 'nunique')).reset_index()
df_dau

In [ ]:
df_dau.plot.line(x='day', y='DAU', title='DAU')
plt.show()

## Анализ пользовательской активности

Дополнительно оценим ежедневную и недельную активность пользователей с помощью метрик DAU и WAU.

In [ ]:
#Рассчитайте и визуализируйте WAU (по оси Х - недели) (исключите неполные недели)

df_wau_0 = df_good.groupby(['week']).agg(WAU = ('id_client', 'nunique')).reset_index()
print(df_wau_0)

In [ ]:
first_full_week = df_good[df_good['day'] == 1]['week'].min()
print(first_full_week)
last_full_week = df_good[df_good['day'] == 7]['week'].max()
print(last_full_week)

In [ ]:
df_wau = df_wau_0[df_wau_0['week']!=5]
df_wau

In [ ]:
df_wau.plot.line(x='week', y='WAU', title='WAU')
plt.show()

In [ ]:
#Соедините DAU и WAU и визуализируйте их на одном графике (по оси Х - дни)

df_dau['week'] = df_dau['dtime_event'].dt.isocalendar().week
print(df_dau)


In [ ]:
dau_wau = df_dau.merge(df_wau, on = 'week', how = 'inner')
dau_wau

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dau_wau['dtime_event'], dau_wau['DAU'], 'b', label = 'DAU')
ax.set_title('Динамика DAU и WAU')
ax.set_xlabel('Дни')
ax.set_ylabel('Кол-во пользователей')
ax.legend(loc='lower left')
ax.grid()

ax2 = ax.twinx()
ax2.plot(dau_wau['dtime_event'], dau_wau['WAU'], 'g', label = 'WAU')
ax2.legend(loc='lower right')

# Итоговые выводы

- Выполнена очистка данных и исключены пользователи, попавшие в несколько экспериментальных групп.
- Рассчитана конверсия в просмотр контента для тестовой и контрольной групп.
- Проведён t-тест для проверки статистической значимости различий.
- Статистически значимого улучшения (или улучшение обнаружено — если таков результат) не выявлено.
- Проанализирована динамика DAU и WAU.теста.